In [1]:
import pandas as pd
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [2]:
# ────────────────────────────────────────────────────────────────────────────────
# 0) Hyperparameters & Constants
# ────────────────────────────────────────────────────────────────────────────────
MAX_VOCAB_SIZE    = 20000
MAX_SEQUENCE_LEN  = 200
EMBEDDING_DIM     = 300
LSTM_UNITS        = 64
BATCH_SIZE        = 64
EPOCHS            = 1
AUTOTUNE          = tf.data.AUTOTUNE
NUM_CLASSES       = 4
CLASS_NAMES       = ["World", "Sports", "Business", "Sci/Tech"]


In [3]:
train_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/train.csv"
test_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/test.csv"

MODEL_DIR = "/Users/sameerkhan/Desktop/sameerkhan/weights/nlp/transformer"
os.makedirs(MODEL_DIR, exist_ok=True)

CHECKPOINT_FILE = f"agnews_transformer_cnn.h5"

FINAL_MODEL_FILE = "agnews_transformer_cnn.keras"

VOCAB_FILE = MODEL_DIR + "/agnews_vocab.txt"

GLOVE_FILE = '/Users/sameerkhan/Desktop/sameerkhan/data/nlp/glove.42B.300d.txt'

In [4]:
train_df = pd.read_csv(train_path,header=0)
train_df = train_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})

test_df = pd.read_csv(test_path,header=0)
test_df = test_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})


# zero-based labels
train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"]  = test_df["label"].astype(int) - 1

# combine title + description
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

In [5]:
# ────────────────────────────────────────────────────────────────────────────────
# 2) Train/validation split
# ────────────────────────────────────────────────────────────────────────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].values,
    train_df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"].values
)
test_texts  = test_df["text"].values
test_labels = test_df["label"].values


In [6]:
# ────────────────────────────────────────────────────────────────────────────────
# 3) TextVectorization
# ────────────────────────────────────────────────────────────────────────────────
vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LEN
)
vectorizer.adapt(train_texts)

def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    token_ids = vectorizer(text)
    return tf.squeeze(token_ids, axis=0), label

def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(len(texts), seed=42)
    ds = ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_texts, train_labels, shuffle=True)
val_ds   = make_dataset(val_texts,   val_labels)
test_ds  = make_dataset(test_texts,  test_labels)

In [7]:
embeddings_index = {}
glovefile = open(GLOVE_FILE,'r',encoding='utf-8')
for line in tqdm(glovefile):
    values = line.split(" ")
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
glovefile.close()

print('Found %s word vectors.' % len(embeddings_index))

1917494it [01:38, 19450.42it/s]

Found 1917494 word vectors.


In [8]:
# 1) Build the embedding matrix from your GloVe dict and vectorizer vocab
vocab = vectorizer.get_vocabulary()  # list length ≥ MAX_VOCAB_SIZE
vocab = vocab[:MAX_VOCAB_SIZE]       # truncate to exactly MAX_VOCAB_SIZE
embedding_matrix = np.zeros((MAX_VOCAB_SIZE, EMBEDDING_DIM), dtype="float32")

for idx, word in enumerate(vocab):
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
    # else leave zeros (or add small random noise)

In [9]:
text_inputs = layers.Input(shape=(MAX_SEQUENCE_LEN,),name="input_tokens", dtype="int32")
embedding_layer = layers.Embedding(input_dim=MAX_VOCAB_SIZE, 
                                   output_dim=EMBEDDING_DIM,
                                   input_length=MAX_SEQUENCE_LEN, 
                                   weights=[embedding_matrix], 
                                   trainable=False,
                                   mask_zero =True)
positional_embedding_layer = layers.Embedding(input_dim=MAX_SEQUENCE_LEN, output_dim=EMBEDDING_DIM, trainable=True)
embedded_sequences = embedding_layer(text_inputs)

positions = tf.range(start=0, limit=MAX_SEQUENCE_LEN, delta=1)
positions = positional_embedding_layer(positions)

embedded_sequences = embedded_sequences + positions


num_heads = 2
ff_dim =256

attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=EMBEDDING_DIM)
attn_out = attn(embedded_sequences, embedded_sequences)
attn_out = layers.Dropout(0.5)(attn_out)
attn_out_f = layers.LayerNormalization(axis=-1)(embedded_sequences + attn_out)

ffn_out = layers.Dense(ff_dim, activation="relu")(attn_out_f)
ffn_out = layers.Dense(EMBEDDING_DIM)(ffn_out)
ffn_out = layers.Dropout(0.5)(ffn_out)
ffn_out_f = layers.LayerNormalization(axis=-1)(attn_out_f + ffn_out)

conv41 = layers.Conv1D(filters=128, kernel_size=16, activation="relu")(ffn_out_f)
pool41 = layers.MaxPooling1D()(conv41)
norm41 = layers.LayerNormalization(axis=-1)(pool41)
conv42 = layers.Conv1D(filters=256, kernel_size=16, activation="relu")(norm41)

trans_pool1 = layers.GlobalAveragePooling1D()(conv42)
trans_d = layers.Dense(256, activation="relu")(trans_pool1)
text_features = layers.Dropout(0.5, name="text_features")(trans_d)

text_features = layers.Dense(256, activation="relu")(text_features)
text_features = layers.LayerNormalization(axis=-1)(text_features)
text_out = layers.Dense(NUM_CLASSES, activation="softmax", name="text_out")(text_features)

model = Model(inputs=[text_inputs], outputs = [text_out] )




/Users/sameerkhan/Desktop/sameerkhan/venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 300)  │  6,000,000 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 200, 300)  │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 200, 300)  │    722,100 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 200, 300)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 200, 300)  │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 200, 300)  │        600 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 200, 256)  │     77,056 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 200, 300)  │     77,100 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 200, 300)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 200, 300)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 200, 300)  │        600 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 185, 128)  │    614,528 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 92, 128)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 92, 128)   │        256 │ max_pooling1d[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 77, 256)   │    524,544 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv1d_1[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_features       │ (None, 256)       │          0 │ dense_2[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 8,149,908 (31.09 MB)

 Trainable params: 2,149,908 (8.20 MB)

 Non-trainable params: 6,000,000 (22.89 MB)

In [11]:
# ────────────────────────────────────────────────────────────────────────────────
# 5) Train
# ────────────────────────────────────────────────────────────────────────────────
ckpt = callbacks.ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, CHECKPOINT_FILE),
    monitor="val_accuracy",
    save_best_only=True
)
es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[ckpt, es]
)

 196/1500 ━━━━━━━━━━━━━━━━━━━━ 33:59 2s/step - accuracy: 0.3721 - loss: 1.4031

KeyboardInterrupt: 

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# 6) Evaluate
# ────────────────────────────────────────────────────────────────────────────────
loss, acc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

119/119 [==============================] - 36s 304ms/step - loss: 0.3010 - accuracy: 0.8997
Test accuracy: 0.8997
